<h1>
    <center>
        BV Regime NN Pipeline<br><br>Starter to Run the Pipeline (train + predict) from a Notebook
    </center>
</h1>

----
- Laique M. Djeutchouang

---

# Overview

This notebook shows how to drive the **whole** pipeline (train + predict) from inside the notebook. Everything on `analysis.ipynb` consumed a model somebody else trained, while this notebook breakdowns the routes the pipeline itself could be run, on a dataset of your choosing, without leaving the notebook.

## What the model produces.

It never assigns a single regime outright. For
every `(time, lat, lon)` cell it emits a probability distribution over the BV
regimes, from which four fields are derived:

| Variable | Dims | Meaning |
| --- | --- | --- |
| `regime_prob` | `(time, lat, lon, regime)` | Probability of each regime |
| `regime_entropy` | `(time, lat, lon)` | Normalised Shannon entropy — `0` confident, `1` maximally ambiguous |
| `regime_pred` | `(time, lat, lon)` | Most likely regime (argmax) |
| `regime_confidence` | `(time, lat, lon)` | Probability of that most likely regime |

The two that matter together are `regime_pred` and `regime_entropy`: the regime
map is only trustworthy where entropy is low, and the high-entropy regions are
themselves a result — they mark transitional dynamics.

## Prerequisites.

The `slvp_nn` environment (see
[`environment.yml`](environment.yml) and the *Environment* section of
[`README.md`](README.md)). Start the kernel from it:

```bash
mamba activate slvp_nn
python -m ipykernel install --user --name slvp_nn --display-name "Python (slvp_nn)"
```

# Setup Requirements

## Import Packages

In [27]:
import os
import sys
import json
import subprocess
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

# pipeline.py lives next to this notebook; fall back to the repo path when the
# notebook is opened from somewhere else.
NN_DIR = Path.cwd() if (Path.cwd() / "pipeline.py").exists() else Path("/home/djeutsch/Projects/seaLevelRegimes/nn")
if str(NN_DIR) not in sys.path:
    sys.path.insert(0, str(NN_DIR))

import pipeline
from pipeline import (BVBRegimeMLP, ModelConfig, TrainConfig,
                      open_dataset, BVB_TERMS, LABEL_VAR)

pipeline.setup_logging(verbose=False)   # the pipeline logs to stdout

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": 0.3, "figure.autolayout": True})

# Zarr 3 warns on every consolidated write; the pipeline does that per block.
warnings.filterwarnings("ignore", message=".*Consolidated metadata.*")

# Cartopy makes the maps geographic; everything still works without it.
# Catch broadly: a cartopy that is installed but misconfigured (no PROJ data
# directory, say) raises at import time with something other than ImportError.
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    from cartopy.mpl.gridliner import LongitudeFormatter, LatitudeFormatter
    ccrs.PlateCarree()          # touch PROJ now rather than mid-figure
    HAS_CARTOPY = True
except Exception as _err:
    HAS_CARTOPY = False
    print(f"cartopy unavailable ({type(_err).__name__}) - falling back to lat/lon axes")

print(f"pipeline   : {NN_DIR / 'pipeline.py'}")
print(f"BVB terms  : {BVB_TERMS}")
print(f"label var  : {LABEL_VAR}")
print(f"cartopy    : {'yes' if HAS_CARTOPY else 'no (plain lat/lon axes)'}")

pipeline   : /home/djeutsch/Projects/seaLevelRegimes/nn/pipeline.py
BVB terms  : ['beta_V', 'BPT', 'Mass_flux', 'eta_dt', 'Curl_dudt', 'Curl_taus', 'Curl_taub', 'Curl_Adv', 'Curl_diff']
label var  : bvb_regime
cartopy    : yes


## Some Customized Mapping Tools

In [24]:
### Define plotting properties 
from string import ascii_lowercase as asci

def number_figures(axes, pos=None, labels=asci, braces=True, **text_kwargs):

    depth = lambda L: isinstance(L, list) and max(map(depth, L)) + 1

    if pos is None:
        pos = [[0.99, 0.93]] * len(axes)
    elif (depth(pos) == 1) & (len(pos) == 2):
        pos = [list(pos)] * len(axes)
    elif (depth(pos) != 2) & (len(pos) != len(axes)):
        raise (Exception, 'check the position is the right format')
    
    for c, ax in enumerate(axes):
        x0, x1 = ax.get_xlim()
        y0, y1 = ax.get_ylim()

        w = x1 - x0
        h = y1 - y0
        x = x0 + w * pos[c][0]
        y = y0 + h * pos[c][1]

        t = '%s' % labels[c]
        if braces:
            t = '(%s)' % (t)
        ax.text(x, y, t, **text_kwargs)
          

def add_map_features(ax, draw_countries=False, draw_labels=False, right=False):
    # Add coastlines
    ax.coastlines(color="lightgrey", linewidth=0.1125)

    # Add land and states
    ax.add_feature(cfeature.LAND, facecolor='darkgrey', edgecolor='lightgrey', linewidth=0.15, zorder=-1)
    if draw_countries:
        ax.add_feature(cfeature.BORDERS, linewidth=0.125, alpha=0.25, edgecolor='black')
    
    # Create gridlines and specify the coordinate system
    gl = ax.gridlines(crs=ccrs.PlateCarree(), dms=True, draw_labels=draw_labels,
                      linewidth=0.15, color='black', alpha=0.25, linestyle='--')
    
    if draw_labels:
        # Customize the labels format
        gl.xformatter = LongitudeFormatter()
        gl.yformatter = LatitudeFormatter()
        
        # Optional: Customize label appearance
        gl.xlabel_style = {'size': 8, 'color': 'black'}
        gl.ylabel_style = {'size': 8, 'color': 'black'}
        
        # Remove default labels on the top and right sides
        gl.top_labels = False
        gl.right_labels = right


def add_grid_borders(ax, land=False, borders=True, draw_labels=True, right=False):
    # Create gridlines and specify the coordinate system
    gl = ax.gridlines(crs=ccrs.PlateCarree(), dms=True, draw_labels=draw_labels,
                      linewidth=0.15, color='black', alpha=0.25, linestyle='--')
    if land:
        ax.add_feature(cfeature.LAND, facecolor='darkgrey', edgecolor='lightgrey', linewidth=0.15, zorder=-1)
        
    if borders:
        ax.add_feature(cfeature.BORDERS, linewidth=0.125, alpha=0.25, edgecolor='black')
    
    if draw_labels:
        # Customize the labels format
        gl.xformatter = LongitudeFormatter()
        gl.yformatter = LatitudeFormatter()
        
        # Optional: Customize label appearance
        gl.xlabel_style = {'size': 8, 'color': 'black'}
        gl.ylabel_style = {'size': 8, 'color': 'black'}
        
        # Remove default labels on the top and right sides
        gl.top_labels = False
        gl.right_labels = right



## Configuration

Everything the notebook needs is set here. `BASE_DIR` mirrors `SLVP_BASE_DIR`,
the same root `pipeline.py` and `submit.sh` build their defaults from.

In [63]:
BASE_DIR = Path(os.environ.get("SLVP_BASE_DIR", "/group/maikesgrp/laique/PPAN/CM4X/NN4X"))

# Features + labels, as written for the pipeline.
INPUT_ZARR = BASE_DIR / "inputs" / "global_NN4X_p25_monthly_features_nc15.zarr"

# Where the Slurm runs put their outputs.
OUTDIR = BASE_DIR / "outputs" / "nn"

# Run to analyse. TAG selects the checkpoint and the prediction store.
TAG = "bvb_mlp_h256x128x64x32_k15_2005_2011"
CKPT = OUTDIR / "models" / f"{TAG}.pt"
PRED_STORE = OUTDIR / "predictions" / f"{TAG}_predictions.zarr"

# Inference window: None keeps the whole record.
PREDICT_YEARS = None            # e.g. ("2012", "2014")

DEVICE = "auto"                 # "auto" | "cpu" | "cuda" | "cuda:0"

# Somewhere to write the notebook's own artefacts.
SCRATCH = Path(os.environ.get("SLVP_NB_SCRATCH", BASE_DIR / "outputs/notebook_outputs"))
SCRATCH.mkdir(parents=True, exist_ok=True)

for name, p in [("input", INPUT_ZARR), ("checkpoint", CKPT),
                ("predictions", PRED_STORE), ("scratch", SCRATCH)]:
    print(f"{name:12s} {'OK     ' if p.exists() else 'missing'}  {p}")

input        OK       /group/maikesgrp/laique/PPAN/CM4X/NN4X/inputs/global_NN4X_p25_monthly_features_nc15.zarr
checkpoint   OK       /group/maikesgrp/laique/PPAN/CM4X/NN4X/outputs/nn/models/bvb_mlp_h256x128x64x32_k15_2005_2011.pt
predictions  OK       /group/maikesgrp/laique/PPAN/CM4X/NN4X/outputs/nn/predictions/bvb_mlp_h256x128x64x32_k15_2005_2011_predictions.zarr
scratch      OK       /group/maikesgrp/laique/PPAN/CM4X/NN4X/outputs/notebook_outputs


# BV Regime NN Pipeline: the Running Routes 

There are three routes, and they differ in what you get back rather than in what
they compute:

| Route | Runs where | You get back | Use it for |
| --- | --- | --- | --- |
| **A — Python API** | This kernel | Live `BVBRegimeMLP`, in-memory `xr.Dataset` | Experiments, custom loops, anything you want to poke at |
| **B — `pipeline.main()`** | This kernel | The job's exact on-disk layout | Reproducing a run, or a short job you would rather not queue |
| **C — `submit.sh`** | Slurm | A job id | Real training runs, the full record, anything on a GPU |

> **A notebook kernel is a bad place for a long run.** Routes A and B block the
> kernel for the whole of training and die with it — a dropped SSH connection
> takes the run with it. Anything beyond a few minutes belongs in route C.

The cells below are written to run cheaply in demo mode and to *print* what they
would do otherwise, so running the notebook top to bottom never starts a real
training run by accident. Flip the `RUN_*` flags deliberately.

## Route A — the Python API

The most direct route: build a `ModelConfig` and a `TrainConfig`, call `.fit()`,
then `.predict()`. Nothing touches disk unless you ask it to, and the fitted
estimator stays in the notebook for further work.

`fit` splits the record **chronologically** (`train_frac`), never randomly, and
fits the scaler on the training months alone — so the validation score is not
contaminated by the months it is scored on.

In [ ]:
RUN_ROUTE_A = DEMO        # set True to run for real

model_cfg = ModelConfig(
    features=list(BVB_TERMS),
    label_var=LABEL_VAR,
    n_regimes=N_REGIMES_DEMO if DEMO else 15,
    hidden=(64, 32) if DEMO else (256, 128, 64, 32, 16),
    rare_regimes=True,        # SiLU activations, better on the rare regimes
    dropout=0.0,
)

train_cfg = TrainConfig(
    epochs=8 if DEMO else 100,
    batch_size=16384 if DEMO else 8192,
    lr=1e-3,
    train_frac=0.7,           # first 70% of months train, last 30% validate
    patience=6 if DEMO else 16,
    class_weights="balanced", # "balanced" | "curriculum" | "none"
    lambda_entropy=0.25,      # entropy's weight in the stopping metric
    seed=42,
)

if RUN_ROUTE_A:
    ds_fit = open_dataset(INPUT_ZARR, model_cfg.features, model_cfg.label_var,
                          years=None if DEMO else ("2005", "2011"))

    est = BVBRegimeMLP(model_cfg, device=DEVICE)
    est.fit(ds_fit, train_cfg)

    ckpt_a = SCRATCH / "route_a" / "models" / "route_a.pt"
    est.save(ckpt_a)
    pipeline.save_history(est.history, ckpt_a.with_name("route_a_history.csv"))

    # Inference on the same dataset - swap the path to predict on another.
    ds_pred = open_dataset(INPUT_ZARR, est.config.features, label_var=None)
    pred_a = est.predict(ds_pred, entropy_unit="fraction")   # in memory
    ds_fit.close(); ds_pred.close()

    print(f"\nfinal val balanced accuracy: {est.history['val_balanced_accuracy'][-1]:.4f}")
    display(pred_a)
else:
    print("RUN_ROUTE_A is False - nothing was trained.")
    print(f"Would train {model_cfg.hidden} on {INPUT_ZARR} for {train_cfg.epochs} epochs.")

For a long record, swap `est.predict(ds)` for `est.predict_to_store(ds, path)`:
identical results, but streamed in blocks of months so memory stays flat.

```python
est.predict_to_store(ds_pred, SCRATCH / "route_a_predictions.zarr",
                     entropy_unit="fraction", time_chunk=12)
pred_a = xr.open_zarr(SCRATCH / "route_a_predictions.zarr", chunks=None)
```

## Route B — `pipeline.main()` with an argument list

`main` is the same entry point `pipeline.sh` calls, so this reproduces a Slurm
run exactly, including the on-disk layout:

```
<outdir>/models/<tag>.pt                        weights + scaler + architecture + history
<outdir>/models/<tag>_history.csv               per-epoch metrics
<outdir>/models/<tag>_config.json               full run configuration
<outdir>/predictions/<tag>_predictions.zarr     the four output fields
```

Pass the flags as a **list of strings** — the same ones `python pipeline.py`
takes, and `pipeline.build_parser().print_help()` is the complete reference.

Runs are idempotent: an existing checkpoint or prediction store is skipped with
a log message rather than recomputed. Pass `--overwrite` to force the work, or
use a fresh `--tag`.

In [ ]:
RUN_ROUTE_B = DEMO        # set True to run for real

argv = [
    "--input", str(INPUT_ZARR),
    "--outdir", str(SCRATCH / "route_b"),
    "--tag", "route_b",
    "--mode", "train-predict",           # "train" | "predict" | "train-predict"
    "--n-regimes", str(N_REGIMES_DEMO if DEMO else 15),
    "--hidden", "64,32" if DEMO else "256,128,64,32,16",
    "--epochs", "6" if DEMO else "100",
    "--batch-size", "16384",
    "--class-weights", "balanced",
    "--entropy-unit", "fraction",
    "--pred-format", "zarr",
    "--device", DEVICE,
    "--overwrite",
]
if not DEMO:
    argv += ["--train-years", "2005", "2011"]

print("equivalent shell command:\n  python pipeline.py " + " ".join(argv), end="\n\n")

if RUN_ROUTE_B:
    status = pipeline.main(argv)
    print(f"\npipeline.main returned {status}")
    assert status == 0, "the pipeline reported a failure"
    print("\nwritten:")
    for p in sorted((SCRATCH / "route_b").rglob("*")):
        if p.is_file() or p.suffix == ".zarr":
            print("  ", p.relative_to(SCRATCH))
else:
    print("RUN_ROUTE_B is False - nothing was run.")

### Inference only, from an existing checkpoint

The common follow-up: a model is already trained, and you want it applied to a
record it never saw. `--mode predict` with `--checkpoint`, and a `--tag` that
does not collide with the training run's outputs.

In [ ]:
predict_argv = [
    "--input", str(INPUT_ZARR),              # only used to locate defaults
    "--predict-input", str(INPUT_ZARR),      # the dataset to actually score
    "--outdir", str(SCRATCH / "route_b"),
    "--mode", "predict",
    "--checkpoint", str(CKPT),
    "--tag", "route_b_predict_only",
    "--entropy-unit", "fraction",
    "--predict-time-chunk", "12",
    "--device", DEVICE,
    "--overwrite",
]
if PREDICT_YEARS:
    predict_argv += ["--predict-years", *PREDICT_YEARS]

print("equivalent shell command:\n  python pipeline.py " + " ".join(predict_argv), end="\n\n")

if RUN_ROUTE_B:
    assert pipeline.main(predict_argv) == 0
    out = SCRATCH / "route_b" / "predictions" / "route_b_predict_only_predictions.zarr"
    display(xr.open_zarr(out, chunks=None))
else:
    print("RUN_ROUTE_B is False - nothing was run.")

In [ ]:
# The full option reference, straight from the parser.
pipeline.build_parser().print_help()

## Route C — submit to Slurm

For anything real. `submit.sh` consumes the job options, turns them into
`sbatch` flags, and forwards everything else to `pipeline.py` untouched.

`--dry-run` prints the command instead of submitting it, which is the safe way
to check an invocation from a notebook.

In [ ]:
def submit(*args, dry_run=True):
    """Call submit.sh. Returns the CompletedProcess; dry_run only prints sbatch.

    Note `--input`: submit.sh checks the dataset exists *before* it honours
    --dry-run, so a dry run still needs a path that is really there.
    """
    cmd = ["bash", str(NN_DIR / "submit.sh"), "--input", str(INPUT_ZARR), *map(str, args)]
    if dry_run:
        cmd.append("--dry-run")
    res = subprocess.run(cmd, capture_output=True, text=True, cwd=NN_DIR)
    print(res.stdout or res.stderr)
    if res.returncode != 0:
        print(f"[submit.sh exited {res.returncode}]")
    return res

# What a real training submission would look like - printed, not submitted.
submit("--cpus", "16", "--mem", "300G", "--time", "12:00:00",
       "--hidden", "256,128,64,32,16",
       "--epochs", "300",
       "--class-weights", "balanced",
       dry_run=True);

In [ ]:
# A GPU run, again dry.
submit("--gpus", "1", "--batch-size", "32768", "--epochs", "300",
       "--tag", "gpu_run", dry_run=True);

# To submit for real, drop dry_run - then watch it:
#   submit("--gpus", "1", "--epochs", "300", dry_run=False)
#   !squeue -u $USER
#   !tail -f dumps/nn/nn_*.out

### Coming back to a submitted run

The job writes to the layout in route B, so re-entering this notebook after it
finishes is just a matter of pointing the configuration cell at it:

```python
TAG   = "bvb_mlp_h256x128x64x32x16_k15_2005_2011"   # or your --tag
CKPT  = OUTDIR / "models" / f"{TAG}.pt"
PRED_STORE = OUTDIR / "predictions" / f"{TAG}_predictions.zarr"
```

and re-running from section 3. Nothing is recomputed: the checkpoint is loaded
and the prediction store is opened as it stands.

---

## Where to go next

* `--class-weights curriculum --curriculum-warmup 15` ramps the balancing in
  gradually. Worth trying when the balanced run sacrifices too much on the
  common regimes — compare the per-regime table in section 7.
* `--n-regimes` must cover the labels; the pipeline raises with the required
  value if it is too small.
* `--entropy-unit percent` rescales the entropy to `[0, 100]`. The analysis
  above reads the unit from the store's attributes, so it stays correct either
  way.
* If the entropy fails the test in `analysis.ipynb` — accuracy flat across the deciles
  — the probabilities are not calibrated for this run, and no entropy threshold
  will fix the maps. Retrain before interpreting them.